# 03) Baseline Models and Feature Selection

In [1]:
import json

import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

train = pd.read_csv("data/train.csv", parse_dates=["date"])
with open("data/columns.json") as f:
    columns = json.load(f)

TARGET_COL = columns["target_cols"][0]
FEATURE_COLS = columns["feature_cols"]

print(f"train: {train.shape[0]} rows, {len(FEATURE_COLS)} features, target = {TARGET_COL!r}")

train: 8443 rows, 33 features, target = 'target_bps_vs_mid'


## Expanding-window cross-validation split

A single fixed fit/eval split gives one point estimate of eval MSE, entirely dependent on which
handful of days happened to land in eval -- not something to pick a penalty on with confidence.
Expanding-window CV rolls the eval block forward across multiple folds: each fold trains on every
day strictly before its eval block (never after -- no lookahead) and the training window grows as
the boundary moves forward. `data/test.csv` is still untouched; all folds live inside
`data/train.csv`.

Walk-forward, one day at a time: fold 0 trains on the first `MIN_TRAIN_DAYS` days and evals on the
next single day; each later fold folds that eval day into training and evals on the day after it.
`MIN_TRAIN_DAYS` is the warm-up -- how much history a fold needs before it's trusted to fit at all.

In [2]:
unique_dates = np.sort(train["date"].unique())
MIN_TRAIN_DAYS = 7
DELTA_DAYS = 1

folds = []
for fold_idx, eval_start in enumerate(range(MIN_TRAIN_DAYS, len(unique_dates), DELTA_DAYS)):
    fit_dates = unique_dates[:eval_start]
    eval_dates = unique_dates[eval_start:eval_start + DELTA_DAYS]
    assert fit_dates.max() < eval_dates.min(), "eval dates must all fall after fit dates"

    fit_mask = train["date"].isin(fit_dates)
    eval_mask = train["date"].isin(eval_dates)
    folds.append((fit_mask, eval_mask))

    print(f"fold {fold_idx}: fit {fit_mask.sum():>4} rows ({len(fit_dates)} days, "
          f"{pd.Timestamp(fit_dates.min()).date()} to {pd.Timestamp(fit_dates.max()).date()}) | "
          f"eval {eval_mask.sum():>4} rows ({len(eval_dates)} days, "
          f"{pd.Timestamp(eval_dates.min()).date()} to {pd.Timestamp(eval_dates.max()).date()})")

fold 0: fit 3476 rows (7 days, 2025-03-03 to 2025-03-11) | eval  496 rows (1 days, 2025-03-12 to 2025-03-12)
fold 1: fit 3972 rows (8 days, 2025-03-03 to 2025-03-12) | eval  497 rows (1 days, 2025-03-13 to 2025-03-13)
fold 2: fit 4469 rows (9 days, 2025-03-03 to 2025-03-13) | eval  497 rows (1 days, 2025-03-14 to 2025-03-14)
fold 3: fit 4966 rows (10 days, 2025-03-03 to 2025-03-14) | eval  497 rows (1 days, 2025-03-17 to 2025-03-17)
fold 4: fit 5463 rows (11 days, 2025-03-03 to 2025-03-17) | eval  496 rows (1 days, 2025-03-18 to 2025-03-18)
fold 5: fit 5959 rows (12 days, 2025-03-03 to 2025-03-18) | eval  497 rows (1 days, 2025-03-19 to 2025-03-19)
fold 6: fit 6456 rows (13 days, 2025-03-03 to 2025-03-19) | eval  496 rows (1 days, 2025-03-20 to 2025-03-20)
fold 7: fit 6952 rows (14 days, 2025-03-03 to 2025-03-20) | eval  497 rows (1 days, 2025-03-21 to 2025-03-21)
fold 8: fit 7449 rows (15 days, 2025-03-03 to 2025-03-21) | eval  497 rows (1 days, 2025-03-24 to 2025-03-24)
fold 9: fit 7

## Ridge regression, tuned over the expanding-window folds

Same pipeline as before -- median-impute (only `near_far_spread_bps` has nulls) -> standardize ->
Ridge, both fit on each fold's `fit` side only. Same alpha grid, wide enough to bracket where eval
MSE bottoms out. For each alpha, metrics are computed per fold on both sides of the split, then
averaged across folds:

- **MSE** in bps^2 (the target is in bps); `eval_mse_std` shows how much the mean is worth trusting.
- **R^2** -- fraction of target variance explained. In-sample R^2 is on the same rows the model was
  fit on; out-of-sample R^2 can go negative if the model does worse than predicting the fold's own
  mean.
- **IC** -- Spearman rank correlation between predicted and actual `target_bps_vs_mid`. This is the
  more standard cross-sectional signal-quality metric in practice: it only cares about getting the
  *ranking* of rows right, not the scale of the prediction, which R^2/MSE both penalize.

In [3]:
alphas = [0.1, 1.0, 10.0, 100.0, 1_000.0, 10_000.0, 100_000.0, 1_000_000.0]
results = []

for alpha in alphas:
    fit_mses, eval_mses = [], []
    fit_r2s, eval_r2s = [], []
    fit_ics, eval_ics = [], []
    for fit_mask, eval_mask in folds:
        X_fit, y_fit = train.loc[fit_mask, FEATURE_COLS], train.loc[fit_mask, TARGET_COL]
        X_eval, y_eval = train.loc[eval_mask, FEATURE_COLS], train.loc[eval_mask, TARGET_COL]

        model = Pipeline([
            ("impute", SimpleImputer(strategy="median")),
            ("scale", StandardScaler()),
            ("ridge", Ridge(alpha=alpha)),
        ])
        model.fit(X_fit, y_fit)

        pred_fit = model.predict(X_fit)
        pred_eval = model.predict(X_eval)

        fit_mses.append(mean_squared_error(y_fit, pred_fit))
        eval_mses.append(mean_squared_error(y_eval, pred_eval))
        fit_r2s.append(r2_score(y_fit, pred_fit))
        eval_r2s.append(r2_score(y_eval, pred_eval))
        fit_ics.append(spearmanr(y_fit, pred_fit).statistic)
        eval_ics.append(spearmanr(y_eval, pred_eval).statistic)

    results.append({
        "alpha": alpha,
        "fit_mse": np.mean(fit_mses),
        "eval_mse_mean": np.mean(eval_mses),
        "eval_mse_std": np.std(eval_mses),
        "fit_r2": np.mean(fit_r2s),
        "eval_r2": np.mean(eval_r2s),
        "fit_ic": np.mean(fit_ics),
        "eval_ic": np.mean(eval_ics),
    })

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

    alpha   fit_mse  eval_mse_mean  eval_mse_std   fit_r2   eval_r2   fit_ic  eval_ic
      0.1 41.664945      51.351139     24.666447 0.432739  0.160676 0.559225 0.526858
      1.0 41.720336      51.327455     24.707764 0.431997  0.161141 0.558193 0.526457
     10.0 41.746532      51.287298     24.675986 0.431643  0.162233 0.558356 0.527945
    100.0 41.774223      50.943364     24.125824 0.431271  0.169152 0.562614 0.533880
   1000.0 42.542451      49.141101     20.974691 0.420926  0.209777 0.584039 0.559202
  10000.0 51.281207      51.204703     27.106135 0.302787  0.239139 0.597611 0.571548
 100000.0 68.214985      66.313702     43.193744 0.072731  0.063526 0.591933 0.561160
1000000.0 72.937191      71.384219     47.802741 0.008412 -0.001728 0.588530 0.557336
